# 06. 멀티 에이전트 아키텍처 설계

멀티에이전트 패턴 + SubAgent + 서브그래프

---

### 학습 목표
1. 멀티에이전트 아키텍처 패턴(Supervisor, Handoff, Swarm)의 **특성과 사용 시점**을 이해할 수 있다
2. Deep Agents의 **SubAgent dict**와 **CompiledSubAgent**로 서브에이전트를 정의하고 실행할 수 있다
3. **서브그래프**와 **namespace 키**를 활용하여 에이전트 간 상태 공유 및 컨텍스트 격리를 구현할 수 있다

### 진행 방식
- 각 섹션의 코드를 순서대로 실행하며 개념을 확인합니다
- `[핵심]` 주석에서 각 코드의 동작 원리를 확인합니다
- Part 1: 멀티에이전트 패턴 + SubAgent 정의 (Part 1)
- Part 2: 상태 공유 + 서브그래프 모듈화 (Part 2)

## 환경 설정

> **`.env` 파일 설정**
> 노트북과 같은 디렉토리에 `.env` 파일 생성 후 아래 내용 작성:
> ```
> OPENAI_API_KEY=sk-...
> LANGFUSE_PUBLIC_KEY=pk-lf-...   # 트레이싱 사용 시
> LANGFUSE_SECRET_KEY=sk-lf-...
> ```

In [1]:
# [L06-1] : 라이브러리 설치
# [핵심] 멀티 에이전트 + Deep Agents SubAgent에 필요한 전체 패키지
# [주의] 버전 충돌 시 런타임 재시작 후 재실행
!uv pip install -q langchain langchain-core langchain-openai langgraph deepagents python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


D:\2026_04_SK하이닉스_AI에이전트프레임워크구현\배포버전\.venv\Scripts\python.exe: No module named pip


In [1]:
# [L06-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True -> 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."

In [2]:
# [L06-3] : 공통 LLM 모델 초기화
# [핵심] 노트북 전체에서 재사용할 모델을 한 곳에서 관리
# [주의] temperature=0 -> 결정적 출력; gpt-4.1-mini로 비용 절감

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
print(f"모델 초기화 완료: {llm.model_name}")

모델 초기화 완료: gpt-4.1-mini


---

# Part 1: 멀티에이전트 패턴 + SubAgent

## [1-1] : 멀티에이전트 아키텍처 개요

**단일 에이전트의 한계**: 도구가 많아지면 도구 선택 정확도가 떨어지고, 프롬프트가 비대해지며, 다른 도메인의 컨텍스트가 섞여 성능이 저하된다.

**멀티 에이전트**는 역할을 분리하고 조정 메커니즘을 도입하여 이 한계를 극복한다.

### 5가지 멀티 에이전트 패턴

| 패턴 | 라우팅 주체 | 상태 공유 | 적합한 상황 |
|------|-----------|----------|------------|
| **Subagents** | 메인 에이전트 | 도구로 격리 | 병렬 처리, 분산 개발 |
| **Handoffs** | `Command` 객체 | 상태 전환 | 순차적 멀티홉 |
| **Skills** | 단일 에이전트 | 프롬프트 교체 | 도메인 특화 |
| **Router** | 분류기 | 병렬 실행 | 멀티 도메인 |
| **Swarm** | 각 에이전트 자율 | P2P | 분산 자율 협업 |

### 패턴 선택 가이드

1. 에이전트가 독립적으로 작업 가능한가? -> **Subagents**
2. 대화 상태가 에이전트 간에 전달되어야 하는가? -> **Handoffs**
3. 하나의 에이전트가 여러 역할을 전환하면 되는가? -> **Skills**
4. 입력을 분류한 뒤 적절한 처리기로 보내면 되는가? -> **Router**
5. 에이전트가 자율적으로 협업해야 하는가? -> **Swarm**

## [1-2] : Supervisor 패턴

**Supervisor(감독자) 패턴**은 중앙 에이전트가 서브에이전트를 도구처럼 호출하여 작업을 위임하는 3계층 아키텍처이다.

| 계층 | 역할 | 특징 |
|------|------|------|
| **저수준 도구** | 외부 서비스 직접 호출 | 단순한 함수 래퍼 |
| **서브에이전트** | 도메인별 추론 + 도구 조합 | 전문 시스템 프롬프트, 독립적 도구 세트 |
| **감독자** | 작업 분해, 위임, 결과 집계 | 전체 대화 기억, 서브에이전트 = 도구로 취급 |

### 핵심 특성
- **중앙 집중 제어** — 모든 라우팅이 감독자를 통해 흐름
- **컨텍스트 격리** — 서브에이전트는 깨끗한 컨텍스트에서 실행
- **병렬 실행** — 여러 서브에이전트를 한 턴에서 동시 호출 가능

In [3]:
# [L06-4] : 저수준 도구 정의 — 서브에이전트가 사용할 외부 서비스 래퍼
# [핵심] 하나의 도구는 하나의 기능만 담당 (단일 책임 원칙)
# [주의] docstring은 LLM이 도구 선택 시 참조 — 명확하게 작성 필수

from langchain_core.tools import tool


@tool
def search_technical_docs(query: str) -> str:
    """기술 문서 데이터베이스에서 관련 문서를 검색합니다."""
    docs_db = {
        "HBM": "HBM3E는 TSV 기반 수직 적층 구조로 1.18TB/s 대역폭을 제공합니다.",
        "DRAM": "DDR5 DRAM은 4800~8400 MT/s 데이터 전송률을 지원합니다.",
        "NAND": "3D NAND는 200단 이상 적층으로 면적 효율을 극대화합니다.",
    }
    for key, val in docs_db.items():
        if key in query:
            return val
    return f"'{query}'에 대한 검색 결과가 없습니다."


@tool
def analyze_data(data_description: str) -> str:
    """데이터를 분석하고 통계 요약을 반환합니다."""
    return f"'{data_description}' 분석 완료: 평균 수율 95.3%, 불량률 0.7%"


@tool
def generate_report(title: str, content: str) -> str:
    """분석 결과를 기반으로 보고서를 생성합니다."""
    return f"보고서 '{title}' 생성 완료 — 내용: {content[:100]}"


print(f"저수준 도구 3개 정의 완료: {[t.name for t in [search_technical_docs, analyze_data, generate_report]]}")

저수준 도구 3개 정의 완료: ['search_technical_docs', 'analyze_data', 'generate_report']


In [4]:
# [L06-5] : 서브에이전트 생성 + 도구 래핑 + Supervisor 조립
# [핵심] create_agent로 전문 프롬프트 + 도구를 가진 에이전트 생성 -> @tool로 래핑
# [주의] 래핑 함수 내부에서 invoke 후 마지막 메시지의 content만 반환 — 컨텍스트 격리

from langchain.agents import create_agent

# 1. 서브에이전트 생성
search_agent = create_agent(
    model=llm,
    tools=[search_technical_docs],
    system_prompt="당신은 기술 문서 검색 전문가입니다. 관련 기술 문서를 검색하고 핵심 정보를 요약하세요.",
    name="search_agent",
)

analysis_agent = create_agent(
    model=llm,
    tools=[analyze_data],
    system_prompt="당신은 데이터 분석 전문가입니다. 데이터를 분석하고 통계적 인사이트를 도출하세요.",
    name="analysis_agent",
)

report_agent = create_agent(
    model=llm,
    tools=[generate_report],
    system_prompt="당신은 보고서 작성 전문가입니다. 분석 결과를 기반으로 구조적인 보고서를 작성하세요.",
    name="report_agent",
)

In [5]:
# 2. 서브에이전트를 도구로 래핑
@tool("search_docs", description="기술 문서를 검색하고 관련 정보를 반환합니다")
def call_search(query: str) -> str:
    """검색 에이전트에 작업을 위임합니다."""
    result = search_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content  # [핵심] 마지막 응답만 반환 -> 컨텍스트 격리


@tool("analyze", description="데이터를 분석하고 통계 요약을 반환합니다")
def call_analysis(query: str) -> str:
    """분석 에이전트에 작업을 위임합니다."""
    result = analysis_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


@tool("write_report", description="분석 결과를 기반으로 보고서를 작성합니다")
def call_report(query: str) -> str:
    """보고서 에이전트에 작업을 위임합니다."""
    result = report_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

In [6]:
# 3. Supervisor 조립
SYSTEM_PROMPT = (
        "당신은 반도체 기술 팀의 감독 에이전트입니다.\n"
        "복잡한 요청을 하위 작업으로 분해하고 적절한 전문 에이전트에 위임하세요:\n"
        "- 기술 문서 검색이 필요하면 search_docs를 사용\n"
        "- 데이터 분석이 필요하면 analyze를 사용\n"
        "- 보고서 작성이 필요하면 write_report를 사용\n"
        "모든 서브에이전트의 결과를 종합하여 사용자에게 일관된 응답을 제공하세요."
    )

supervisor = create_agent(
    model=llm,
    tools=[call_search, call_analysis, call_report],
    system_prompt=SYSTEM_PROMPT
)

In [7]:
# [L06-6] : Supervisor 실행 테스트
# [핵심] 감독자가 검색 -> 분석 순으로 서브에이전트를 호출하여 종합 응답 생성
# [주의] 서브에이전트 간 의존성이 있을 때 감독자가 순서를 조정

response = supervisor.invoke({"messages": [{"role": "user", "content": "HBM 메모리 기술 동향을 조사하고 분석해주세요."}]})
print("=== Supervisor 응답 ===")
print(response["messages"][-1].content[:500])

=== Supervisor 응답 ===
HBM 메모리 기술 동향과 최신 데이터 분석 결과를 종합한 보고서를 작성했습니다.

보고서 요약:
- HBM은 고대역폭, 저전력 소비를 특징으로 하며, 3D 스태킹과 TSV 기술 발전으로 성능과 용량이 크게 향상되고 있습니다.
- 최신 HBM3E는 1.18TB/s의 대역폭을 제공하며, 평균 수율 95.3%, 불량률 0.7%로 안정적인 생산이 이루어지고 있습니다.
- 대역폭은 최대 819GB/s에 달하며, 전력 효율은 1.2W/Gbps 수준으로 고성능 컴퓨팅과 데이터 센터에 적합합니다.
- 앞으로 인공지능, 빅데이터, 고성능 컴퓨팅 분야에서 HBM 수요가 증가하며 기술 발전이 가속화될 전망입니다.

필요하시면 더 구체적인 기술 세부사항이나 시장 동향 분석도 추가로 제공해 드릴 수 있습니다.


In [8]:
response

{'messages': [HumanMessage(content='HBM 메모리 기술 동향을 조사하고 분석해주세요.', additional_kwargs={}, response_metadata={}, id='5d5d88fc-a1b6-4960-bc39-f8ee54b748a2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 210, 'total_tokens': 269, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_9865f4ebfa', 'id': 'chatcmpl-Drt6XsD4Z7rH4Cd7I9YjGWNhC5192', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed7b7-4f34-77c0-b76c-2fa012e92454-0', tool_calls=[{'name': 'search_docs', 'args': {'query': 'HBM 메모리 기술 동향'}, 'id': 'call_xj6mjce9mP1AvDGS9VS0F9aw', 'type': 'tool_call'}, {'name': 'analyze', 'args': {'query': 'HBM 메모리 관련 최신 데이터 

## [1-3] : Handoff 패턴

**Handoff 패턴**은 `Command(goto=...)` 또는 `Command(update={...})`로 에이전트 간 **상태를 전환**하는 아키텍처이다. Supervisor가 중앙 집중형이라면, Handoff는 **분산형** 제어 흐름이다.

| 특성 | Supervisor | Handoff |
|------|-----------|--------|
| **제어 흐름** | 중앙 집중 (감독자) | 분산 (에이전트 간 직접 전환) |
| **상태 공유** | 도구 결과만 반환 | 전체 대화 상태 전달 |
| **적합한 상황** | 병렬 처리, 독립 작업 | 순차적 멀티홉, 워크플로 |

In [9]:
# [L06-7] : Handoff 전환 도구 + 에이전트 정의
# [핵심] Command(goto=...)로 StateGraph 내 다른 에이전트 노드로 전환
# [주의] graph=Command.PARENT는 서브그래프가 아닌 부모 그래프에서 전환하겠다는 의미

from langgraph.types import Command
from langgraph.graph import StateGraph, START, END, MessagesState


# 전환 도구
@tool
def transfer_to_analyst() -> Command:
    """대화를 데이터 분석 전문가에게 전환합니다."""
    return Command(goto="analyst", graph=Command.PARENT)


@tool
def transfer_to_researcher() -> Command:
    """대화를 기술 조사 전문가에게 전환합니다."""
    return Command(goto="researcher", graph=Command.PARENT)


@tool
def provide_answer(answer: str) -> str:
    """사용자에게 최종 답변을 제공합니다."""
    return answer


# 에이전트 노드 정의
router_agent = create_agent(
    model=llm,
    tools=[transfer_to_analyst, transfer_to_researcher],
    system_prompt=(
        "당신은 안내 데스크입니다. 사용자의 요청을 분석하여 적절한 전문가에게 전환하세요:\n"
        "- 데이터 분석, 수율, 통계 관련 -> analyst에게 전환\n"
        "- 기술 조사, 문서 검색, 동향 분석 -> researcher에게 전환"
    )
)

researcher_agent = create_agent(
    model=llm,
    tools=[search_technical_docs, provide_answer],
    system_prompt="당신은 반도체 기술 연구원입니다. 기술 문서를 검색하여 정확한 답변을 제공하세요."
)

analyst_agent = create_agent(
    model=llm,
    tools=[analyze_data, provide_answer],
    system_prompt="당신은 반도체 데이터 분석가입니다. 데이터를 분석하고 통계적 인사이트를 제공하세요."
)

print("Handoff 에이전트 3개 정의 완료: router, researcher, analyst")

Handoff 에이전트 3개 정의 완료: router, researcher, analyst


In [10]:
# [L06-8] : Handoff 그래프 조립 및 실행
# [핵심] StateGraph에 에이전트를 노드로 등록하고 START -> router 연결
# [주의] 전환 도구가 Command를 반환하면 LangGraph가 자동으로 goto 노드로 이동

handoff_builder = StateGraph(MessagesState)
handoff_builder.add_node("router", router_agent)
handoff_builder.add_node("researcher", researcher_agent)
handoff_builder.add_node("analyst", analyst_agent)
handoff_builder.add_edge(START, "router")    # 진입점: 라우터

handoff_graph = handoff_builder.compile()

# 실행 테스트
result = handoff_graph.invoke({"messages": [{"role": "user", "content": "HBM 메모리의 대역폭 특성을 알려주세요."}]})
print("=== Handoff 결과 ===")
print(result["messages"][-1].content[:300])

=== Handoff 결과 ===
HBM 메모리는 TSV(Through-Silicon Via) 기반의 수직 적층 구조를 사용하여 매우 높은 대역폭을 제공합니다. 예를 들어, 최신 HBM3E 메모리는 약 1.18TB/s의 대역폭을 지원하여 고성능 컴퓨팅 및 그래픽 처리에 적합합니다.


In [11]:
result

{'messages': [HumanMessage(content='HBM 메모리의 대역폭 특성을 알려주세요.', additional_kwargs={}, response_metadata={}, id='9b1e708b-681f-43bf-8806-be8943c2342d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 113, 'total_tokens': 138, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_7c5d442c61', 'id': 'chatcmpl-Drt6wZFpvaymvIZwCEoJdCJS4YboI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed7b7-b886-77d0-b906-e4fb9c75ae91-0', tool_calls=[{'name': 'search_technical_docs', 'args': {'query': 'HBM 메모리 대역폭 특성'}, 'id': 'call_eGWILbxlpEzJXZVQSH9V03g9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tok

## [1-4] : Deep Agents SubAgent 정의

**Deep Agents**의 `SubAgent`는 딕셔너리 형태로 서브에이전트를 정의한다. LangGraph의 `create_agent`와 달리, Deep Agents는 메인 에이전트가 **자동으로** 서브에이전트를 도구처럼 호출하고 컨텍스트를 격리한다.

### SubAgent 필수 필드

| 필드 | 타입 | 설명 |
|------|------|------|
| `name` | `str` | 고유 식별자 |
| `description` | `str` | 역할 설명 (메인 에이전트가 호출 판단에 사용) |
| `system_prompt` | `str` | 서브에이전트 전용 지침 |
| `tools` | `list` | 사용할 도구 목록 |

### 컨텍스트 블로트(Context Bloat) 해결

에이전트가 도구를 사용할 때마다 입력/출력이 컨텍스트에 쌓인다. 서브에이전트는 독립 컨텍스트에서 실행되어 **요약 결과만** 메인 에이전트에 반환하므로, 컨텍스트가 깔끔하게 유지된다.

In [12]:
# [L06-9] : Deep Agents SubAgent dict 정의
# [핵심] name/description/system_prompt/tools 4개 필드로 서브에이전트 선언
# [주의] description은 메인 에이전트가 호출 판단에 사용 -> 명확하게 작성

from deepagents import create_deep_agent

# 리서치 서브에이전트
research_subagent = {
    "name": "researcher",
    "description": "기술 문서를 검색하고 핵심 정보를 요약합니다. 기술 조사가 필요한 질문에 사용하세요.",
    "system_prompt": (
        "당신은 전문 기술 리서처입니다.\n"
        "기술 문서를 검색하고 핵심만 추출하여 간결하게 요약합니다.\n"
        "결과는 항상 한국어로 작성하며, 500단어 이내로 요약하세요."
    ),
    "tools": [search_technical_docs],
}

# 분석 서브에이전트
analysis_subagent = {
    "name": "analyst",
    "description": "데이터를 분석하고 통계적 인사이트를 추출합니다.",
    "system_prompt": (
        "당신은 데이터 분석 전문가입니다.\n"
        "제공된 데이터에서 패턴, 트렌드, 핵심 인사이트를 추출합니다.\n"
        "분석 결과를 불릿 포인트로 정리하세요."
    ),
    "tools": [analyze_data],
}

print(f"SubAgent 정의 완료: {research_subagent['name']}, {analysis_subagent['name']}")
print(f"researcher 설명: {research_subagent['description'][:50]}...")

SubAgent 정의 완료: researcher, analyst
researcher 설명: 기술 문서를 검색하고 핵심 정보를 요약합니다. 기술 조사가 필요한 질문에 사용하세요....


In [13]:
# [L06-10] : SubAgent를 포함하는 메인 에이전트 생성 및 실행
# [핵심] create_deep_agent()에 subagents 리스트를 전달하면 자동으로 도구 래핑
# [주의] 메인 에이전트가 description을 보고 어떤 서브에이전트를 호출할지 자동 결정

main_agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "당신은 프로젝트 매니저입니다.\n"
        "사용자의 요청을 분석하고, 필요하면 전문 서브에이전트에게 작업을 위임합니다.\n"
        "서브에이전트의 결과를 종합하여 최종 답변을 작성합니다.\n"
        "한국어로 응답하세요."
    ),
    subagents=[research_subagent, analysis_subagent],
)

result = main_agent.invoke({"messages": [{"role": "user", "content": "HBM 메모리 기술을 조사하고 분석해주세요."}]})

print("=== Deep Agents SubAgent 응답 ===")
print(result["messages"][-1].content[:500])

=== Deep Agents SubAgent 응답 ===
HBM(High Bandwidth Memory) 메모리 기술은 TSV(Through-Silicon Via) 기반의 수직 적층 구조를 통해 매우 높은 대역폭을 제공하는 차세대 메모리 기술입니다. 최신 버전인 HBM3E는 최대 1.18TB/s의 대역폭을 지원하여 기존 메모리 대비 월등한 성능을 자랑합니다.

주요 내용은 다음과 같습니다.

1. 기술 개념
- 여러 개의 DRAM 다이를 수직으로 적층하고 TSV 기술로 다이 간 전기 신호를 전달
- 메모리 칩과 프로세서 간 데이터 전송 거리를 줄이고 병렬 데이터 전송 극대화

2. 특징
- TSV 기반 수직 적층 구조로 공간 효율성 우수
- 매우 높은 대역폭 (HBM3E 기준 1.18TB/s)
- 낮은 전력 소비와 발열 감소
- 신호 지연 최소화

3. 장점
- 기존 DDR 메모리 대비 대역폭 수 배 이상
- 전력 효율 뛰어남
- 소형화된 시스템 설계 가능

4. 단점
- 제조 공정 복잡, 비용 높음
- TSV 적층 기술 난이도 높아 수율


## [1-5] : CompiledSubAgent + context_schema

미리 컴파일된 **LangGraph 그래프**를 서브에이전트로 사용할 수 있다. 복잡한 워크플로(조건 분기, 반복 등)가 필요한 경우에 유용하다.

### CompiledSubAgent vs SubAgent dict

| 항목 | SubAgent dict | CompiledSubAgent |
|------|--------------|------------------|
| **정의 방식** | 필드 4개 (name, description, ...) | LangGraph 그래프 + runnable |
| **사용 시점** | 단순 도구 조합 | 조건 분기, 반복 루프, 복잡 워크플로 |
| **유연성** | 제한적 | 완전한 그래프 제어 |

### context_schema
`context_schema`를 정의하면 런타임 컨텍스트를 서브에이전트에 자동 전파할 수 있다.

In [17]:
# [L06-11] : CompiledSubAgent — LangGraph 그래프를 서브에이전트로 래핑
# [핵심] 기존 LangGraph 그래프를 runnable 키로 전달하여 서브에이전트화
# [주의] CompiledSubAgent는 name, description, runnable 3개 키 사용

from deepagents import CompiledSubAgent, create_deep_agent

# LangGraph 그래프로 만든 커스텀 분석 에이전트
custom_analysis_graph = create_deep_agent(
    model=llm,
    tools=[analyze_data],
    system_prompt="당신은 데이터 분석 전문가입니다. 데이터를 수집하고 통계적으로 분석하여 인사이트를 도출합니다.",
)

# CompiledSubAgent로 래핑
data_analyst_subagent: CompiledSubAgent = {
    "name": "data-analyst",
    "description": "데이터를 수집하고 분석하여 통계적 인사이트를 제공합니다.",
    "runnable": custom_analysis_graph,
}

print(f"CompiledSubAgent 정의 완료: {data_analyst_subagent['name']}")

CompiledSubAgent 정의 완료: data-analyst


In [19]:
# [L06-12] : context_schema를 활용한 런타임 컨텍스트 전파
# [핵심] context_schema로 구조를 정의하고, config의 context 키로 값을 전달
# [주의] 모든 서브에이전트에 컨텍스트가 자동 전파됨

from langchain_core.messages import HumanMessage

context_agent = create_deep_agent(
    model=llm,
    system_prompt="사용자 맞춤 어시스턴트입니다. 한국어로 응답하세요.",
    subagents=[data_analyst_subagent],
    context_schema={"user_id": str, "language": str},
)

# 컨텍스트와 함께 실행
result = context_agent.invoke(
    {"messages": [HumanMessage(content="다음 월별 매출 데이터를 분석해 주세요. 데이터: [120, 135, 128, 160, 172, 168, 190, 205, 198, 220, 245, 260]")]},
    config={
        "context": {
            "user_id": "user-123",
            "language": "ko",
        },
    },
)

print("=== context_schema 활용 결과 ===")
print(result["messages"][-1].content[:300])

=== context_schema 활용 결과 ===
월별 매출 데이터 [120, 135, 128, 160, 172, 168, 190, 205, 198, 220, 245, 260]에 대해 다음과 같이 분석할 수 있습니다.

1. 총 매출: 12개월간 매출 합계
2. 평균 매출: 월별 평균 매출
3. 최대 매출: 가장 높은 매출과 해당 월
4. 최소 매출: 가장 낮은 매출과 해당 월
5. 매출 증감 추이: 월별 매출 변화 패턴

계산을 진행하겠습니다.
월별 매출 데이터 분석 결과입니다.

1. 총 매출: 12개월간 매출 합계 = 120 + 135 + 128 + 160 + 172 + 16


---

# Part 2: 상태 공유 + 서브그래프

## [2-1] : 에이전트 간 상태 공유

멀티 에이전트 시스템에서 **에이전트 간 상태 공유**는 핵심 설계 과제이다.

### 상태 공유 4가지 방식

| 방식 | 설명 | 적합한 상황 |
|------|------|------------|
| **공유 키** | 부모-서브그래프 간 동일 키로 매핑 | 스키마가 같을 때 |
| **래퍼 노드** | 래퍼 함수에서 상태를 변환하여 전달 | 스키마가 다를 때 |
| **메시지 전달** | `MessagesState`를 통해 대화 히스토리 공유 | 에이전트 간 대화 |
| **도구 결과** | 서브에이전트가 결과 문자열만 반환 | 컨텍스트 격리 |

In [21]:
# [L06-13] : 공유 키 방식 — 부모와 서브그래프가 동일 키 사용
# [핵심] 부모와 서브그래프의 State에 동일한 키가 있으면 자동 매핑
# [주의] 키 이름이 일치해야 자동 연결됨 — 이름 불일치 시 래퍼 노드 사용

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 서브그래프: 텍스트 분석
class AnalysisState(TypedDict):
    text: str
    word_count: int
    char_count: int


def count_words(state: AnalysisState) -> dict:
    return {"word_count": len(state["text"].split())}


def count_chars(state: AnalysisState) -> dict:
    return {"char_count": len(state["text"])}


analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("words", count_words)
analysis_builder.add_node("chars", count_chars)
analysis_builder.add_edge(START, "words")
analysis_builder.add_edge("words", "chars")
analysis_builder.add_edge("chars", END)
analysis_subgraph = analysis_builder.compile()

# 서브그래프 단독 테스트
sub_result = analysis_subgraph.invoke({"text": "LangGraph에서 보내는 인사"})
print(f"서브그래프 단독 테스트: 단어={sub_result['word_count']}, 문자={sub_result['char_count']}")

서브그래프 단독 테스트: 단어=3, 문자=18


## [2-2] : 서브그래프로 에이전트 모듈화

**서브그래프**는 독립적으로 컴파일된 그래프를 부모 그래프의 노드로 사용하는 것이다.

### 서브그래프의 장점
- **모듈화** — 복잡한 워크플로를 작은 단위로 분리
- **재사용** — 동일 서브그래프를 여러 부모 그래프에서 활용
- **팀별 개발** — 서브그래프 단위로 독립적 개발 가능

### 두 가지 통합 패턴
1. **공유 키 패턴** — 부모와 서브그래프의 키가 동일할 때 `add_node("name", subgraph)` 직접 전달
2. **래퍼 노드 패턴** — 스키마가 다를 때 래퍼 함수에서 상태 변환 후 서브그래프 호출

In [22]:
# [L06-14] : 공유 키 패턴 — 서브그래프를 부모 그래프의 노드로 직접 등록
# [핵심] add_node에 컴파일된 서브그래프를 전달 — 공유 키로 자동 매핑
# [주의] 부모 State에 서브그래프의 모든 키가 포함되어야 함

class PipelineState(TypedDict):
    text: str
    word_count: int
    char_count: int
    summary: str


def create_summary(state: PipelineState) -> dict:
    return {
        "summary": f"텍스트는 {state['word_count']}개의 단어와 {state['char_count']}개의 문자를 포함합니다."
    }


parent_builder = StateGraph(PipelineState)
parent_builder.add_node("analyze", analysis_subgraph)  # [핵심] 서브그래프를 노드로 직접 등록
parent_builder.add_node("summarize", create_summary)
parent_builder.add_edge(START, "analyze")
parent_builder.add_edge("analyze", "summarize")
parent_builder.add_edge("summarize", END)

pipeline = parent_builder.compile()

result = pipeline.invoke({"text": "LangGraph는 강력한 상태 기반 에이전트 워크플로를 가능하게 합니다"})
print(f"공유 키 패턴 결과: {result['summary']}")

공유 키 패턴 결과: 텍스트는 8개의 단어와 40개의 문자를 포함합니다.


In [23]:
# [L06-15] : 래퍼 노드 패턴 — 스키마가 다른 서브그래프 통합
# [핵심] 부모 상태 -> 서브그래프 입력 변환 -> 실행 -> 결과를 부모 상태로 매핑
# [주의] 래퍼 함수가 상태 변환의 핵심 — 부모/서브 스키마 간 매핑을 명확히 정의

# 서브그래프: 감성 분석 (부모와 다른 스키마)
class SentimentState(TypedDict):
    log_entry: str
    sentiment: str
    confidence: float


def analyze_sentiment(state: SentimentState) -> dict:
    """규칙 기반 감성 분석"""
    text = state["log_entry"].lower()
    positive_words = {"good", "great", "excellent", "happy", "love", "amazing"}
    negative_words = {"bad", "terrible", "awful", "hate", "poor", "horrible"}
    pos = sum(1 for w in text.split() if w in positive_words)
    neg = sum(1 for w in text.split() if w in negative_words)
    total = pos + neg
    if total == 0:
        return {"sentiment": "neutral", "confidence": 0.5}
    elif pos > neg:
        return {"sentiment": "positive", "confidence": round(pos / total, 2)}
    else:
        return {"sentiment": "negative", "confidence": round(neg / total, 2)}


sub_builder = StateGraph(SentimentState)
sub_builder.add_node("analyze", analyze_sentiment)
sub_builder.add_edge(START, "analyze")
sub_builder.add_edge("analyze", END)
sentiment_subgraph = sub_builder.compile()


# 부모 그래프: 다른 스키마
class FeedbackState(TypedDict):
    customer_name: str
    feedback_text: str
    analysis_result: str


def call_sentiment_subgraph(state: FeedbackState) -> dict:
    """래퍼: 부모 상태 -> 서브그래프 실행 -> 결과 매핑"""
    # [핵심] 부모 -> 서브그래프: 상태 변환
    sub_input = {"log_entry": state["feedback_text"]}
    sub_output = sentiment_subgraph.invoke(sub_input)
    # [핵심] 서브그래프 -> 부모: 결과 매핑
    return {"analysis_result": f"{sub_output['sentiment']} (confidence: {sub_output['confidence']})"}


def format_report(state: FeedbackState) -> dict:
    return {"analysis_result": f"[보고서] {state['customer_name']}: {state['analysis_result']}"}


parent_builder = StateGraph(FeedbackState)
parent_builder.add_node("sentiment", call_sentiment_subgraph)
parent_builder.add_node("report", format_report)
parent_builder.add_edge(START, "sentiment")
parent_builder.add_edge("sentiment", "report")
parent_builder.add_edge("report", END)
feedback_pipeline = parent_builder.compile()

# 실행 테스트
r1 = feedback_pipeline.invoke({"customer_name": "Alice", "feedback_text": "The product is great and amazing"})
r2 = feedback_pipeline.invoke({"customer_name": "Bob", "feedback_text": "The experience was terrible and poor"})
print(f"래퍼 노드 결과 1: {r1['analysis_result']}")
print(f"래퍼 노드 결과 2: {r2['analysis_result']}")

래퍼 노드 결과 1: [보고서] Alice: positive (confidence: 1.0)
래퍼 노드 결과 2: [보고서] Bob: negative (confidence: 1.0)


## [2-3] : Namespace 기반 컨텍스트 격리

Deep Agents에서 **네임스페이스 키**를 사용하면 특정 서브에이전트에만 전달되는 설정을 추가할 수 있다.

### 네임스페이스 키 형식

```python
config = {
    "context": {
        "user_id": "user-123",             # 모든 에이전트에 전파
        "researcher:max_depth": 3,          # researcher에만 전달
        "data-analyst:strict_mode": True,   # data-analyst에만 전달
    }
}
```

### 동작 원리
- `콜론(:)` 없는 키 -> 모든 에이전트에 전파
- `에이전트이름:키` 형식 -> 해당 서브에이전트에만 전달
- 서브에이전트는 자신의 네임스페이스 키만 볼 수 있음 -> 컨텍스트 격리

In [24]:
# [L06-16] : Namespace 키 기반 서브에이전트별 컨텍스트 격리
# [핵심] "에이전트이름:키" 형식으로 특정 서브에이전트에만 설정 전달
# [주의] 콜론 없는 키는 모든 에이전트에 전파

# 서브에이전트 2개 정의
collector_subagent = {
    "name": "collector",
    "description": "기술 문서를 수집합니다.",
    "system_prompt": "당신은 데이터 수집 전문가입니다. 기술 문서를 검색하고 수집합니다.",
    "tools": [search_technical_docs],
}

writer_subagent = {
    "name": "writer",
    "description": "수집된 정보를 바탕으로 보고서를 작성합니다.",
    "system_prompt": "당신은 테크니컬 라이터입니다. 간결하고 명확한 보고서를 작성합니다.",
    "tools": [],
}

ns_agent = create_deep_agent(
    model=llm,
    system_prompt="사용자 요청을 처리합니다. 한국어로 응답하세요.",
    subagents=[collector_subagent, writer_subagent],
    context_schema={"user_id": str, "output_format": str},
)

result = ns_agent.invoke(
    {"messages": [{"role": "user", "content": "DRAM 기술 동향을 조사하고 보고서를 작성해주세요."}]},
    config={
        "context": {
            "user_id": "user-456",               # 모든 에이전트에 전파
            "collector:max_results": 5,            # collector에만 전달
            "writer:output_format": "markdown",    # writer에만 전달
        },
    },
)

print("=== Namespace 격리 결과 ===")
print(result["messages"][-1].content[:400])

=== Namespace 격리 결과 ===
DRAM 기술 동향 보고서 초안을 작성했습니다. 주요 내용은 다음과 같습니다.

1. DRAM 기본 개념
- 동적 랜덤 액세스 메모리로, 커패시터에 데이터를 저장하며 주기적 리프레시 필요
- 고속 데이터 접근, 대용량 저장, 휘발성 메모리 특성

2. 최근 기술 트렌드
- DDR5: 대역폭 및 용량 향상, 서버 및 고성능 PC 시장 확대
- LPDDR5/LPDDR5X: 모바일용 저전력 고속 메모리
- HBM: 3D 적층 고대역폭 메모리, AI 및 GPU용
- 성능 향상: 데이터 전송 속도 증가, ECC 강화, 인터페이스 개선
- 저전력 기술: 전압 감소, 동적 전력 조절 등
- 3D 적층 기술: TSV 활용, 칩 면적 대비 용량 증가

3. 주요 제조사 동향
- 삼성전자, SK하이닉스, 마이크론 중심으로 D


## [2-4] : 멀티에이전트 파이프라인 실습

지금까지 배운 패턴을 통합하여 **수집 -> 분석 -> 작성** 3단계 파이프라인을 구축한다.

### 파이프라인 아키텍처

```
사용자 질문
  -> [Coordinator] 작업 분해
    -> [data-collector] 정보 수집
    -> [data-analyzer] 데이터 분석
    -> [report-writer] 보고서 작성
  -> [Coordinator] 최종 응답
```

In [25]:
# [L06-17] : 3단계 멀티에이전트 파이프라인 — collector -> analyzer -> writer
# [핵심] 메인 에이전트가 서브에이전트를 순서대로 호출하여 파이프라인 실행
# [주의] 각 서브에이전트는 독립 컨텍스트에서 실행 — 결과 요약만 메인에 반환

pipeline_agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "당신은 프로젝트 코디네이터입니다.\n"
        "사용자의 요청을 분석하고, 적절한 서브에이전트에게 순서대로 작업을 위임합니다:\n"
        "1. 먼저 data-collector로 정보를 수집합니다.\n"
        "2. 수집된 정보를 data-analyzer에게 전달하여 분석합니다.\n"
        "3. 분석 결과를 report-writer에게 전달하여 보고서를 작성합니다.\n"
        "최종 보고서를 사용자에게 전달합니다. 한국어로 응답하세요."
    ),
    subagents=[
        {
            "name": "data-collector",
            "description": "다양한 소스에서 원시 데이터와 정보를 수집합니다.",
            "system_prompt": (
                "당신은 데이터 수집 전문가입니다.\n"
                "주어진 주제에 대해 기술 문서를 검색하고, 관련 데이터를 수집합니다.\n"
                "수집한 데이터를 구조화하여 반환하세요."
            ),
            "tools": [search_technical_docs],
        },
        {
            "name": "data-analyzer",
            "description": "수집된 데이터를 분석하여 핵심 인사이트를 추출합니다.",
            "system_prompt": (
                "당신은 데이터 분석 전문가입니다.\n"
                "제공된 데이터에서 패턴, 트렌드, 핵심 인사이트를 추출합니다.\n"
                "분석 결과를 불릿 포인트로 정리하세요."
            ),
            "tools": [analyze_data],
        },
        {
            "name": "report-writer",
            "description": "분석 결과를 바탕으로 전문적인 보고서를 작성합니다.",
            "system_prompt": (
                "당신은 테크니컬 라이터입니다.\n"
                "분석 결과를 바탕으로 명확하고 읽기 쉬운 보고서를 작성합니다.\n"
                "보고서 구조: 개요 -> 핵심 발견 -> 결론"
            ),
            "tools": [],
        },
    ],
)

print("멀티 서브에이전트 파이프라인 생성 완료")

멀티 서브에이전트 파이프라인 생성 완료


In [26]:
# [L06-18] : 파이프라인 실행
# [핵심] 코디네이터가 collector -> analyzer -> writer 순서로 자동 라우팅
# [주의] 서브에이전트 간 데이터 전달은 메인 에이전트가 중재

pipeline_result = pipeline_agent.invoke({"messages": [{"role": "user", "content": "DRAM과 HBM 기술을 비교 분석한 보고서를 작성해주세요."}]})

print("=== 파이프라인 결과 ===")
print(pipeline_result["messages"][-1].content[:600])

=== 파이프라인 결과 ===
기술 개요 부분을 작성했습니다. 비교 분석 부분도 작성하여 보고서를 완성할 예정입니다. 계속 진행해도 될까요?


In [27]:
# [L06-19] : 실행 흐름 추적 — 메시지 히스토리 분석
# [핵심] 전체 메시지 히스토리를 확인하여 어떤 도구가 어떤 순서로 호출되었는지 추적
# [주의] 프로덕션에서는 LangSmith/Langfuse 트레이싱으로 시각화 권장

print("=== 실행 흐름 추적 ===")
for i, msg in enumerate(pipeline_result["messages"]):
    role = msg.type if hasattr(msg, "type") else "unknown"
    content_preview = str(msg.content)[:80] if hasattr(msg, "content") else ""

    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"  [{i}] {role} -> 도구 호출: {tc['name']}({str(tc['args'])[:60]})")
    elif content_preview:
        print(f"  [{i}] {role}: {content_preview}...")

=== 실행 흐름 추적 ===
  [0] human: DRAM과 HBM 기술을 비교 분석한 보고서를 작성해주세요....
  [1] ai -> 도구 호출: task({'description': 'DRAM과 HBM 기술에 대해 다양한 출처에서 원시 데이터를 수집합니다. 기술)
  [2] tool: 다양한 출처에서 수집한 DRAM과 HBM 기술에 대한 원시 데이터는 다음과 같습니다.

1. DRAM 기술
- 기술 사양: DDR5 DRAM은 ...
  [3] ai -> 도구 호출: task({'description': '수집된 DRAM과 HBM 기술 데이터를 바탕으로 두 기술의 성능, 구조, 응용)
  [4] tool: - DRAM과 HBM은 모두 메모리 기술로서, 각각의 성능과 구조, 응용 분야, 장단점에서 차별화된 특성을 보임.
- 성능: HBM은 DRAM에...
  [5] ai -> 도구 호출: task({'description': '분석된 DRAM과 HBM 기술 비교 결과를 바탕으로 전문적이고 체계적인 보고서)
  [6] tool: 서론

현대 컴퓨팅 시스템에서 메모리 기술은 시스템 성능과 에너지 효율성에 결정적인 영향을 미치는 핵심 요소입니다. 특히, DRAM(Dynami...
  [7] ai -> 도구 호출: task({'description': '보고서 작성 중간 단계입니다. 기술 개요, 비교 분석, 결론 및 향후 전망 부)
  [8] tool: 보고서 작성에 필요한 기술 개요, 비교 분석, 결론 및 향후 전망 부분을 작성하겠습니다. 우선 기술 개요부터 시작하겠습니다.

---

# 기술...
  [9] ai: 기술 개요 부분을 작성했습니다. 비교 분석 부분도 작성하여 보고서를 완성할 예정입니다. 계속 진행해도 될까요?...


## [2-5] : Swarm 패턴 (참고)

**Swarm 패턴**은 중앙 감독자 없이 에이전트들이 **자율적으로** 협업하는 분산형 아키텍처이다.

### Swarm vs Supervisor vs Handoff

| 특성 | Supervisor | Handoff | Swarm |
|------|-----------|--------|-------|
| **제어** | 중앙 집중 | 순차 전환 | 분산 자율 |
| **통신** | 감독자 경유 | 1:1 전환 | P2P |
| **확장성** | 감독자 병목 | 선형 체인 | 수평 확장 |
| **적합 상황** | 명확한 위계 | 순차 워크플로 | 자율 협업 |

### Swarm 구현 핵심

```python
# 각 에이전트가 다른 에이전트의 전환 도구를 보유
agent_a = create_agent(
    model=llm,
    tools=[transfer_to_b, transfer_to_c, own_tools...],
    system_prompt="자율적으로 판단하여 필요 시 다른 에이전트에 전환하세요.",
    name="agent_a",
)
```

### 주의 사항
- Swarm은 에이전트 수가 많아질수록 **전환 도구가 기하급수적으로 증가**
- 무한 루프 방지를 위한 **최대 전환 횟수 제한** 필요
- 디버깅이 어려우므로 **트레이싱 필수**

In [28]:
# [L06-20] : Swarm 패턴 미니 예시 — 에이전트 간 자율 전환
# [핵심] 각 에이전트가 다른 에이전트의 전환 도구를 보유하여 자율적으로 라우팅
# [주의] 무한 루프 방지를 위해 반드시 종료 도구(final_answer) 포함

# 전환 도구 정의
@tool
def swarm_to_tech() -> Command:
    """기술 질문을 기술 전문가에게 전환합니다."""
    return Command(goto="tech_expert", graph=Command.PARENT)


@tool
def swarm_to_biz() -> Command:
    """비즈니스 질문을 비즈니스 전문가에게 전환합니다."""
    return Command(goto="biz_expert", graph=Command.PARENT)


@tool
def final_answer(answer: str) -> str:
    """최종 답변을 제공합니다."""
    return answer


# Swarm 에이전트 — 각자 상대방 전환 도구 보유
tech_expert = create_agent(
    model=llm,
    tools=[search_technical_docs, swarm_to_biz, final_answer],
    system_prompt=(
        "당신은 기술 전문가입니다. 기술 질문에 답변하세요.\n"
        "비즈니스 관련 질문이면 swarm_to_biz로 전환하세요.\n"
        "답변이 준비되면 final_answer를 사용하세요."
    ),
    name="tech_expert",
)

biz_expert = create_agent(
    model=llm,
    tools=[analyze_data, swarm_to_tech, final_answer],
    system_prompt=(
        "당신은 비즈니스 분석가입니다. 비즈니스 질문에 답변하세요.\n"
        "기술 관련 질문이면 swarm_to_tech로 전환하세요.\n"
        "답변이 준비되면 final_answer를 사용하세요."
    ),
    name="biz_expert",
)

# Swarm 그래프 조립
swarm_builder = StateGraph(MessagesState)
swarm_builder.add_node(tech_expert)
swarm_builder.add_node(biz_expert)
swarm_builder.add_edge(START, "tech_expert")  # 진입점

swarm_graph = swarm_builder.compile()

# 테스트: 기술 질문
swarm_result = swarm_graph.invoke(
    {"messages": [{"role": "user", "content": "NAND 메모리의 적층 기술에 대해 알려주세요."}]}
)
print("=== Swarm 패턴 결과 ===")
print(swarm_result["messages"][-1].content[:300])

=== Swarm 패턴 결과 ===
NAND 메모리의 적층 기술은 3D NAND로 대표됩니다. 3D NAND는 메모리 셀을 수직으로 200단 이상 적층하여 면적 효율을 극대화하는 기술입니다. 이를 통해 동일한 면적에서 더 많은 데이터를 저장할 수 있으며, 성능과 내구성도 향상됩니다.


---

## [정리] : 핵심 요약

| 항목 | 내용 |
|------|------|
| **멀티에이전트 패턴** | Supervisor(중앙 집중), Handoff(순차 전환), Swarm(분산 자율) |
| **Supervisor** | `create_agent` + 서브에이전트를 `@tool`로 래핑 -> 감독자가 위임 |
| **Handoff** | `Command(goto=...)` + `StateGraph` -> 에이전트 간 상태 전환 |
| **Deep Agents SubAgent** | dict 기반: `name`, `description`, `system_prompt`, `tools` |
| **CompiledSubAgent** | LangGraph 그래프를 `runnable`로 연결하여 서브에이전트화 |
| **context_schema** | 런타임 컨텍스트를 서브에이전트에 자동 전파 |
| **서브그래프** | 독립 컴파일 그래프를 부모 그래프의 노드로 사용 |
| **공유 키 패턴** | 부모-서브그래프 간 동일 키로 자동 매핑 |
| **래퍼 노드 패턴** | 스키마가 다를 때 래퍼 함수에서 상태 변환 |
| **Namespace 키** | `"에이전트이름:키"` 형식으로 서브에이전트별 컨텍스트 격리 |
| **파이프라인** | collector -> analyzer -> writer 순차 위임 패턴 |
| **설계 원칙** | 단일 책임, 명확한 인터페이스, 최소 도구, 간결한 결과 |